In [1]:
# system
import os
import sys
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT_PATH)

import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

import numpy as np
import plotly.express as px

# personalized modules
from src.utils.dataset import get_raw_transactions

from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer

#models
from sklearn.pipeline import Pipeline
from src.unsupervised_model import AMLAnomalyEnsemble, IsolationForestModel, HDBScanModel

#preprocess
from src.features import FeatureGenerator

2026-04-02 22:59:40.402 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-02 22:59:40.403 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-02 22:59:40.406 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
seed = 42

In [3]:
data_path = f"{ROOT_PATH}\\src\\data\\output\\raw_transactions.csv"
raw_df = get_raw_transactions(data_path)
raw_df = raw_df.sample(n=50000, random_state=seed)

In [4]:
raw_df.shape

(50000, 5)

In [5]:
raw_df.head()

,timestamp,transaction_date,sender_customer,receiver_customer,amount
3969700,2022-09-09 08:15:00,2022-09-09,80C8896A0_219449,80E38F1F0_25960,223.11
2754162,2022-09-06 23:05:00,2022-09-06,802559890_2591,806C1FCF0_24850,"578,814.58"
3147317,2022-09-07 18:39:00,2022-09-07,804B6F670_6010,80C908B00_28693,"2,220.64"
855955,2022-09-02 06:42:00,2022-09-02,81038A6C0_117299,813ABF970_136519,"2,231.34"
1273371,2022-09-02 20:09:00,2022-09-02,80314D3B0_5466,807BE9270_14663,"17,456.40"


In [6]:
feature_generator = FeatureGenerator()
feature_generator.fit(raw_df)
feature_df = feature_generator.transform(raw_df)

In [7]:
feature_df.head()

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,...,transaction_count_received_30d,transaction_count_received_90d,sent_received_ratio,transaction_direction_ratio,total_amount_90d_ratio,total_amount_7d_ratio,total_amount_30d_ratio,transaction_count_90d_ratio,transaction_count_7d_ratio,transaction_count_30d_ratio
0,100428660_70,11.00,"1,828.00","3,524.93","827,874,679.98",167.57,"1,180.09",483.69,"8,858,628.25","1,738.90",...,3.00,0,"234,796.12",152.33,0.00,"179,614.18","272,632.69",0.00,167.12,163.67
1,1004286A8_70,8.00,"1,123.00","2,557.67","546,350,460.12",262.23,957.23,199.89,"8,770,034.16",645.67,...,3.00,0,"213,529.18",124.78,0.00,"94,033.31","495,619.75",0.00,160.00,107.67
2,1004286F0_70,1.00,226.00,310.35,"44,184,098.11",310.35,753.59,0.00,"1,685,915.80",310.35,...,1.00,0,"141,913.02",113.00,0.00,"42,188,346,066,377.14","6,430.73",0.00,"157,000,000.00",69.00
3,100428738_70,0.00,159.00,0.00,"13,176,281.98",0.00,725.08,0.00,"602,246.98",0.00,...,0.00,0,"13,176,281.98",159.00,0.00,"4,740,422,648,219.01","8,435,859,334,316.07",0.00,"110,000,000.00","49,000,000.00"
4,100428780_70,1.00,195.00,158.61,"35,013,086.88",158.61,674.40,0.00,"1,777,710.99",158.61,...,1.00,0,"219,369.37",97.50,0.00,"34,572,322,204,541.51","2,778.96",0.00,"143,000,000.00",52.00


In [8]:
feature_columns = [col for col in feature_df.columns if col != 'customer_id']
X = feature_df[feature_columns]

In [9]:
log_transformer = FunctionTransformer(np.log1p, validate=False)
identity_transformer = FunctionTransformer(lambda x: x, validate=False)

iso_params = {
    "random_state": seed,
    "contamination": 0.05
}

hdbscan_params = {
    'min_cluster_size': 1000,
    'min_samples': 120,
    'metric': 'manhattan',
    'cluster_selection_method': 'eom'
}

pipeline_iforest = Pipeline([
    ("identity", identity_transformer),  # no log
    ("scaler", RobustScaler()),
    ("model", IsolationForestModel(iso_params))
])

pipeline_hdbscan = Pipeline([
    ("log", log_transformer),
    ("scaler", RobustScaler()),
    ("model", HDBScanModel(hdbscan_params))
])

models = [
    pipeline_iforest,
    pipeline_hdbscan
]

In [10]:
ensemble = AMLAnomalyEnsemble(models=models)

In [11]:
ensemble.fit(X)

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [12]:
scores = ensemble.score(X)

c:\Users\ferna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [13]:
fig = px.violin(
    y=scores,
    box=False,
    points=False,
    title="Combined Risk Score Density"
)
fig.show()

In [14]:
feature_df["final_score"] = scores

In [15]:
feature_df[["customer_id", "final_score"]]

,customer_id,final_score
0,100428660_70,0.89
1,1004286A8_70,0.89
2,1004286F0_70,0.88
3,100428738_70,0.86
4,100428780_70,0.87
...,...,...
72986,81495B051_54219,0.29
72987,81495B671_256496,0.54
72988,814962510_17246,0.23
72989,814962A81_53744,0.17
